Per Context Union, For a given 89 instance, if any model was able to compile the test per context type. 

FILES = {
    "class": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/execute_results_pre.json"},
    },
    "method": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/pre/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/execute_results_pre.json"},
    },
    "minimal": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/pre/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/execute_results_pre.json"},
    },
}

In [5]:
#!/usr/bin/env python3
"""
RQ1: Per-context UNION across the 3 models, at BOTH granularities.

For each context, of the 89 instances:

  INSTANCE LEVEL
    compile_inst : # instances where >=1 model COMPILED at least one test
                   (non-empty "compiled" list)
    execute_inst : # instances where >=1 model PASSED at least one test
                   (non-empty "passed" list)

  FILE LEVEL  (distinct test-file names, scoped per instance)
    compile_file : total # of distinct test files that COMPILED under
                   at least one model, summed over instances
    execute_file : total # of distinct test files that PASSED under
                   at least one model, summed over instances

Union = GPT-4o OR Qwen OR GPT-OSS.

Key assumption (file level): identical file names across models, for the
SAME instance, denote the SAME file -> counted once. Valid only if the
U-numbering is consistent across models. See `merge_files_across_models`.
"""

import csv
import json
from collections import defaultdict
from pathlib import Path

OUTPUT_CSV = "union_instance_and_file.csv"  # written to current working directory

# ---------------------------------------------------------------------------
# context -> model -> {"compile": <compile.json>, "execute": <execute.json>}
# ---------------------------------------------------------------------------
FILES = {
    "class": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp7BatchResultsOp2/pre/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp7BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp7BatchResults/pre/execute_results_pre.json"},
    },
    "method": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp6BatchResults/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp6BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp6BatchResults/pre/execute_results_pre.json"},
    },
    "minimal": {
        "GPT-4o":  {"compile": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTResults/Exp3BatchResults/execute_results_pre.json"},
        "Qwen":    {"compile": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/Qwen480Results/Exp3BatchResults/pre/execute_results_pre.json"},
        "GPT-OSS": {"compile": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/compile_results_pre.json",
                    "execute": "/Volumes/Rachna-HD/GPTOSSResults/Exp3BatchResults/pre/execute_results_pre.json"},
    },
}

TOTAL_INSTANCES = 89


def _load(path):
    if not path or not Path(path).exists():
        if path:
            print(f"  [WARN] missing file, skipping: {path}")
        return {}
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def files_by_instance(path, results_key, list_key) -> dict:
    """
    Return {instance_id: set(file_names)} for one model file.

    results_key : "compilation_results" or "execution_results"
    list_key    : "compiled"            or "passed"
    Falls back to a bare top-level dict if the wrapper key is absent.
    """
    blob = _load(path)
    section = blob.get(results_key, blob) or {}
    out = {}
    for inst, d in section.items():
        if not isinstance(d, dict):
            continue
        files = d.get(list_key, [])
        if not isinstance(files, list):
            files = []
        out[inst] = set(files)          # set() of one instance's file names
    return out


def main():
    rows = []

    for ctx, model_files in FILES.items():
        # file-level: per instance, the set of file names hit by ANY model
        # --- THIS is where "same name across models => same file" is enforced.
        compile_files = defaultdict(set)   # inst -> set(file names)
        execute_files = defaultdict(set)

        for paths in model_files.values():
            for inst, fs in files_by_instance(
                    paths.get("compile"), "compilation_results", "compiled").items():
                compile_files[inst] |= fs        # union across models
            for inst, fs in files_by_instance(
                    paths.get("execute"), "execution_results", "passed").items():
                execute_files[inst] |= fs

        # instance-level: an instance counts if it has >=1 file (any model)
        compile_inst = sum(1 for fs in compile_files.values() if fs)
        execute_inst = sum(1 for fs in execute_files.values() if fs)

        # file-level: total distinct files across all instances
        compile_file = sum(len(fs) for fs in compile_files.values())
        execute_file = sum(len(fs) for fs in execute_files.values())

        rows.append((ctx, compile_inst, execute_inst, compile_file, execute_file))
        print(f"{ctx:<10} "
              f"compile[inst={compile_inst:>2}/{TOTAL_INSTANCES}, files={compile_file:>4}]   "
              f"execute[inst={execute_inst:>2}/{TOTAL_INSTANCES}, files={execute_file:>4}]")

    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["context",
                    "compile_instance_union", "execute_instance_union",
                    "compile_file_union", "execute_file_union",
                    "total_instances"])
        for ctx, ci, ei, cf, ef in rows:
            w.writerow([ctx, ci, ei, cf, ef, TOTAL_INSTANCES])
    print(f"\nCSV written to: {Path(OUTPUT_CSV).resolve()}")


if __name__ == "__main__":
    main()

class      compile[inst=42/89, files=2924]   execute[inst=36/89, files=1852]
  [WARN] missing file, skipping: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/compile_results_pre.json
  [WARN] missing file, skipping: /Volumes/Rachna-HD/GPTResults/Exp6BatchResults/execute_results_pre.json
method     compile[inst=38/89, files=1865]   execute[inst=35/89, files=1075]
  [WARN] missing file, skipping: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/compile_results_pre.json
  [WARN] missing file, skipping: /Volumes/Rachna-HD/GPTResults/Exp3BatchResults/execute_results_pre.json
minimal    compile[inst=37/89, files=1775]   execute[inst=32/89, files= 907]

CSV written to: /Users/rachnaraj/Documents/Research/TSE/LLMBBC/NewExpAfterCanary/ResultsForRQs/RQ1/union_instance_and_file.csv
